In [1]:
# ============================================================
# Notebook : 05_dice_bypass.ipynb
# Purpose  : Quantify, across ALL generated counterfactuals (not a single
#            anecdote), how often an UNCONSTRAINED DiCE explanation edits a
#            variable that lies OFF the Step-3 causal path - the failure the
#            framework's causal constraint is designed to prevent. For every
#            modellable category a classifier is trained, unconstrained
#            counterfactuals are generated for positive queries, the top
#            standardized-change variable of each counterfactual is recorded,
#            and the share whose top edit is off-path (the "bypass rate") is
#            reported. The Skin-temp (ST) category - which has no dedicated
#            sensor channel - is highlighted as the illustrative case in which
#            the true driver is unobserved and the edit lands on an unrelated
#            activity/environment channel.
# Design   : Single code cell. No absolute path is printed. Fixed seed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import dice_ml

warnings.filterwarnings("ignore")
# Silence DiCE/tqdm progress bars for clean, reproducible notebook output.
try:
    from functools import partialmethod
    import tqdm as _tqdm_mod
    import tqdm.auto as _tqdm_auto
    _tqdm_mod.tqdm.__init__ = partialmethod(_tqdm_mod.tqdm.__init__, disable=True)
    _tqdm_auto.tqdm.__init__ = partialmethod(_tqdm_auto.tqdm.__init__, disable=True)
except Exception:
    pass
SEED = 20260821
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})
GRAY = ["0.15", "0.40", "0.60", "0.75", "0.88"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
paths = pd.read_pickle(DATA / "causal_paths.pkl")["causal_paths"]
FEATURES = meta["sensor_cont"]
ABN = meta["abnormal_categories"]
sd = df[FEATURES].std().replace(0, 1)

MODELLABLE = [c for c in ABN if c != "ER"]
N_QUERY, N_CF = 40, 5


def train(cat):
    y = df["cat_" + cat].values
    X = df[FEATURES].astype(float)
    groups = df["HH_ID"].values
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X, y, groups))
    clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                                 random_state=SEED, n_jobs=-1).fit(X.iloc[tr], y[tr])
    tr_df = X.iloc[tr].copy(); tr_df["target"] = y[tr]
    te_df = X.iloc[te].copy(); te_df["target"] = y[te]
    return clf, tr_df, te_df


rows, detail_rows = [], []
st_example = None
for cat in MODELLABLE:
    on_path = set(v for v in paths.get(cat, []) if v in FEATURES)
    clf, tr_df, te_df = train(cat)
    d = dice_ml.Data(dataframe=tr_df, continuous_features=FEATURES, outcome_name="target")
    m = dice_ml.Model(model=clf, backend="sklearn")
    exp = dice_ml.Dice(d, m, method="random")
    pos = te_df[te_df["target"] == 1].drop(columns=["target"])
    if len(pos) == 0:
        continue
    q = pos.sample(min(N_QUERY, len(pos)), random_state=SEED).reset_index(drop=True)

    top_vars, offpath_flags = [], []
    for i in range(len(q)):
        base = q.iloc[i]
        try:
            cf = exp.generate_counterfactuals(q.iloc[[i]], total_CFs=N_CF, desired_class="opposite",
                                              features_to_vary="all", verbose=False)
            fdf = cf.cf_examples_list[0].final_cfs_df
        except Exception:
            fdf = None
        if fdf is None or len(fdf) == 0:
            continue
        for _, cfr in fdf.iterrows():
            changes = {v: abs((cfr[v] - base[v]) / sd[v]) for v in FEATURES}
            top = max(changes, key=changes.get)
            top_vars.append(top)
            off = top not in on_path
            offpath_flags.append(off)
            if cat == "ST" and off and st_example is None and changes[top] > 0:
                st_example = {"query_top_change": top,
                              "delta_std": round(changes[top], 3),
                              "on_path": ", ".join(sorted(on_path)) or "(none)"}
    n_cf = len(offpath_flags)
    if n_cf == 0:
        continue
    off_rate = float(np.mean(offpath_flags))
    vc = pd.Series(top_vars).value_counts()
    top_off = next((v for v in vc.index if v not in on_path), "(none)")
    rows.append({"category": cat, "n_counterfactuals": n_cf,
                 "on_path_vars": ", ".join(sorted(on_path)) or "(none)",
                 "bypass_rate_offpath": round(off_rate, 3),
                 "most_frequent_offpath_edit": top_off,
                 "most_frequent_edit_overall": vc.index[0]})
    for v, cnt in vc.items():
        detail_rows.append({"category": cat, "edited_variable": v, "count": int(cnt),
                            "share": round(cnt / n_cf, 3), "on_path": v in on_path})

bypass = pd.DataFrame(rows)
bypass.to_csv(TAB / "table_dice_bypass.csv", index=False, encoding="utf-8-sig")
detail = pd.DataFrame(detail_rows)
detail.to_csv(TAB / "table_dice_bypass_detail.csv", index=False, encoding="utf-8-sig")

# ============================================================
# Figure 9. Off-path (bypass) rate of unconstrained DiCE per category.
# ============================================================
fig, ax = plt.subplots(figsize=(5.6, 3.4))
lab = {"HR": "Heart rate", "BR": "Breath rate", "ST": "Skin temp.", "SP": "SpO2"}
order = bypass.sort_values("bypass_rate_offpath", ascending=False)
ax.bar([lab.get(c, c) for c in order["category"]], order["bypass_rate_offpath"],
       color=GRAY[1], edgecolor="black", linewidth=0.6)
ax.set_ylabel("Share of counterfactuals editing an off-path variable")
ax.set_xlabel("Abnormal category")
ax.set_ylim(0, 1.05)
save_fig(fig, "fig09_dice_bypass_rate")

# ============================================================
# Figure 10. Which variables unconstrained DiCE edits for Skin-temp (ST).
#            Bars shaded by on-path vs off-path.
# ============================================================
st = detail[detail["category"] == "ST"].sort_values("count", ascending=False)
if len(st):
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    colors = [GRAY[3] if on else GRAY[0] for on in st["on_path"]]
    ax.bar(st["edited_variable"], st["count"], color=colors, edgecolor="black", linewidth=0.6)
    ax.set_ylabel("Counterfactuals with top edit")
    ax.set_xlabel("Edited variable (dark = off causal path)")
    plt.xticks(rotation=90, fontsize=7)
    save_fig(fig, "fig10_st_edited_variables")

# ---- Console summary ----
print("=== DiCE causal-path bypass quantification (unconstrained) ===")
print(f"Queries per category: up to {N_QUERY}; counterfactuals per query: {N_CF}")
print()
print(bypass.to_string(index=False))
print()
if st_example:
    print("Illustrative ST (skin-temp) counterfactual:")
    print(f"  on-path variables     : {st_example['on_path']}")
    print(f"  top edited variable   : {st_example['query_top_change']} "
          f"(standardized change {st_example['delta_std']}) -- OFF causal path")
print()
print("Figures saved: fig09_dice_bypass_rate, fig10_st_edited_variables  (png + pdf, 600 dpi)")
print("Tables saved : table_dice_bypass, table_dice_bypass_detail")


=== DiCE causal-path bypass quantification (unconstrained) ===
Queries per category: up to 40; counterfactuals per query: 5

category  n_counterfactuals                 on_path_vars  bypass_rate_offpath most_frequent_offpath_edit most_frequent_edit_overall
      HR                200                  Illuminance                0.810            SkinTemperature            SkinTemperature
      BR                200                  Illuminance                0.830            SkinTemperature            SkinTemperature
      ST                200 Illuminance, SkinTemperature                0.500                       SPO2            SkinTemperature
      SP                200                         SPO2                0.865                Illuminance                Illuminance

Illustrative ST (skin-temp) counterfactual:
  on-path variables     : Illuminance, SkinTemperature
  top edited variable   : SPO2 (standardized change 7.256) -- OFF causal path

Figures saved: fig09_dice_bypass_rat